In [1]:
import pandas as pd
import numpy as np

In [2]:
train = pd.read_csv("data/raw/train.csv")
holidays = pd.read_csv("data/raw/holidays_events.csv")
stores = pd.read_csv("data/raw/stores.csv")
oil = pd.read_csv("data/raw/oil.csv")
transactions = pd.read_csv("data/raw/transactions.csv")

In [3]:
train["date"] = pd.to_datetime(train["date"])
holidays["date"] = pd.to_datetime(holidays["date"])
oil["date"] = pd.to_datetime(oil["date"])
transactions["date"] = pd.to_datetime(transactions["date"])

In [4]:
oil["dcoilwtico"] = oil["dcoilwtico"].ffill().bfill() # Filling oil missing values

## merge stores with train dataset

In [5]:
df = train.merge(stores, on="store_nbr", how="left")

## Merge Transactions with the train dataset

In [6]:
df = df.merge(transactions, on=["date", "store_nbr"], how="left")
df["transactions"] = df["transactions"].fillna(0)

## Merge Oil with the train dataset

In [7]:
df = df.merge(oil, on="date", how="left")

In [8]:
# merge holidays with the train dataset

In [9]:
holiday_flags = holidays[["date"]].drop_duplicates().copy()
holiday_flags["is_holiday"] = 1

df = df.merge(holiday_flags, on="date", how="left")
df["is_holiday"] = df["is_holiday"].fillna(0).astype(int)

In [10]:
df.isnull().sum()

id                   0
date                 0
store_nbr            0
family               0
sales                0
onpromotion          0
city                 0
state                0
type                 0
cluster              0
transactions         0
dcoilwtico      857142
is_holiday           0
dtype: int64

## After merging the oil table into the main modeling dataset, many missing values appeared in `dcoilwtico`
because some sales dates did not have matching oil-price records. 
These values were handled using forward fill and backward fill to preserve the time-series structure.

In [11]:
df["dcoilwtico"] = df["dcoilwtico"].ffill().bfill()

In [12]:
print(df.isnull().sum())

id              0
date            0
store_nbr       0
family          0
sales           0
onpromotion     0
city            0
state           0
type            0
cluster         0
transactions    0
dcoilwtico      0
is_holiday      0
dtype: int64


In [13]:
df.shape

(3000888, 13)

In [14]:
df.head()

,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,transactions,dcoilwtico,is_holiday
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,0.0,93.14,1
1,1,2013-01-01,1,BABY CARE,0.0,0,Quito,Pichincha,D,13,0.0,93.14,1
2,2,2013-01-01,1,BEAUTY,0.0,0,Quito,Pichincha,D,13,0.0,93.14,1
3,3,2013-01-01,1,BEVERAGES,0.0,0,Quito,Pichincha,D,13,0.0,93.14,1
4,4,2013-01-01,1,BOOKS,0.0,0,Quito,Pichincha,D,13,0.0,93.14,1


In [ ]:
## Step 14 — clean the merged dataset

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 13 columns):
 #   Column        Dtype         
---  ------        -----         
 0   id            int64         
 1   date          datetime64[ns]
 2   store_nbr     int64         
 3   family        object        
 4   sales         float64       
 5   onpromotion   int64         
 6   city          object        
 7   state         object        
 8   type          object        
 9   cluster       int64         
 10  transactions  float64       
 11  dcoilwtico    float64       
 12  is_holiday    int32         
dtypes: datetime64[ns](1), float64(3), int32(1), int64(4), object(4)
memory usage: 286.2+ MB


In [16]:
df.head()

,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,transactions,dcoilwtico,is_holiday
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,0.0,93.14,1
1,1,2013-01-01,1,BABY CARE,0.0,0,Quito,Pichincha,D,13,0.0,93.14,1
2,2,2013-01-01,1,BEAUTY,0.0,0,Quito,Pichincha,D,13,0.0,93.14,1
3,3,2013-01-01,1,BEVERAGES,0.0,0,Quito,Pichincha,D,13,0.0,93.14,1
4,4,2013-01-01,1,BOOKS,0.0,0,Quito,Pichincha,D,13,0.0,93.14,1


In [18]:
df.describe(include="all")

,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,transactions,dcoilwtico,is_holiday
count,3.000888e+06,3000888,3.000888e+06,3000888,3.000888e+06,3.000888e+06,3000888,3000888,3000888,3.000888e+06,3.000888e+06,3.000888e+06,3.000888e+06
unique,NaN,NaN,NaN,33,NaN,NaN,22,16,5,NaN,NaN,NaN,NaN
top,NaN,NaN,NaN,AUTOMOTIVE,NaN,NaN,Quito,Pichincha,D,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,90936,NaN,NaN,1000296,1055868,1000296,NaN,NaN,NaN,NaN
mean,1.500444e+06,2015-04-24 08:27:04.703088384,2.750000e+01,NaN,3.577757e+02,2.602770e+00,NaN,NaN,NaN,8.481481e+00,1.555808e+03,6.792490e+01,1.496437e-01
min,0.000000e+00,2013-01-01 00:00:00,1.000000e+00,NaN,0.000000e+00,0.000000e+00,NaN,NaN,NaN,1.000000e+00,0.000000e+00,2.619000e+01,0.000000e+00
25%,7.502218e+05,2014-02-26 18:00:00,1.400000e+01,NaN,0.000000e+00,0.000000e+00,NaN,NaN,NaN,4.000000e+00,9.300000e+02,4.637750e+01,0.000000e+00
50%,1.500444e+06,2015-04-24 12:00:00,2.750000e+01,NaN,1.100000e+01,0.000000e+00,NaN,NaN,NaN,8.500000e+00,1.331000e+03,5.341000e+01,0.000000e+00
75%,2.250665e+06,2016-06-19 06:00:00,4.100000e+01,NaN,1.958473e+02,0.000000e+00,NaN,NaN,NaN,1.300000e+01,1.976250e+03,9.572000e+01,0.000000e+00
max,3.000887e+06,2017-08-15 00:00:00,5.400000e+01,NaN,1.247170e+05,7.410000e+02,NaN,NaN,NaN,1.700000e+01,8.359000e+03,1.106200e+02,1.000000e+00


In [19]:
df.duplicated().sum()

0

## Sorting the dataset

In [20]:
df = df.sort_values(["store_nbr", "family", "date"]).reset_index(drop=True)

In [21]:
df.head()

,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,transactions,dcoilwtico,is_holiday
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,0.0,93.14,1
1,1782,2013-01-02,1,AUTOMOTIVE,2.0,0,Quito,Pichincha,D,13,2111.0,93.14,0
2,3564,2013-01-03,1,AUTOMOTIVE,3.0,0,Quito,Pichincha,D,13,1833.0,92.97,0
3,5346,2013-01-04,1,AUTOMOTIVE,3.0,0,Quito,Pichincha,D,13,1863.0,93.12,0
4,7128,2013-01-05,1,AUTOMOTIVE,5.0,0,Quito,Pichincha,D,13,1509.0,93.12,1


In [22]:
df.dtypes

id                       int64
date            datetime64[ns]
store_nbr                int64
family                  object
sales                  float64
onpromotion              int64
city                    object
state                   object
type                    object
cluster                  int64
transactions           float64
dcoilwtico             float64
is_holiday               int32
dtype: object

## Converting "family", "city", "state", "type" columns to category for better memory and model prep

In [23]:
categorical_cols = ["family", "city", "state", "type"]

for col in categorical_cols:
    df[col] = df[col].astype("category")

In [24]:
df.dtypes

id                       int64
date            datetime64[ns]
store_nbr                int64
family                category
sales                  float64
onpromotion              int64
city                  category
state                 category
type                  category
cluster                  int64
transactions           float64
dcoilwtico             float64
is_holiday               int32
dtype: object

## Safe copy for modelling 

In [25]:
model_df = df.copy()

## checking sales values

In [26]:
model_df["sales"].describe()

count    3.000888e+06
mean     3.577757e+02
std      1.101998e+03
min      0.000000e+00
25%      0.000000e+00
50%      1.100000e+01
75%      1.958473e+02
max      1.247170e+05
Name: sales, dtype: float64

In [27]:
(model_df["sales"] < 0).sum()

0

## inspecting promotion values 

In [28]:
model_df["onpromotion"].describe()

count    3.000888e+06
mean     2.602770e+00
std      1.221888e+01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      7.410000e+02
Name: onpromotion, dtype: float64

In [29]:
model_df["onpromotion"].value_counts().head(20)

onpromotion
0     2389559
1      174551
2       79386
3       45862
4       31659
5       24540
6       22079
7       18917
8       15587
9       13850
10      12540
11      11017
12       9655
13       8458
14       7234
15       6387
16       5850
17       5224
18       4830
19       4469
Name: count, dtype: int64

In [32]:
model_df.duplicated().sum()

0

In [31]:
model_df.to_csv("data/processed/modeling_data_clean.csv", index=False)

## Step 14 — Data cleaning summary

- Checked the merged dataset structure and column types
- Verified missing values were handled
- Checked for duplicate rows
- Sorted the data by store, family, and date
- Converted selected text columns to categorical type
- Created a clean modeling copy for feature engineering
- Saved the cleaned merged dataset for later steps

# Step 15 -> Feature Engineering

## time-based features creation 

In [33]:
model_df["year"] = model_df["date"].dt.year
model_df["month"] = model_df["date"].dt.month
model_df["day"] = model_df["date"].dt.day
model_df["day_of_week"] = model_df["date"].dt.dayofweek
model_df["week_of_year"] = model_df["date"].dt.isocalendar().week.astype(int)
model_df["is_weekend"] = model_df["day_of_week"].isin([5, 6]).astype(int)
model_df["quarter"] = model_df["date"].dt.quarter

In [34]:
model_df[["date", "year", "month", "day", "day_of_week", "week_of_year", "is_weekend", "quarter"]].head()

,date,year,month,day,day_of_week,week_of_year,is_weekend,quarter
0,2013-01-01,2013,1,1,1,1,0,1
1,2013-01-02,2013,1,2,2,1,0,1
2,2013-01-03,2013,1,3,3,1,0,1
3,2013-01-04,2013,1,4,4,1,0,1
4,2013-01-05,2013,1,5,5,1,1,1


In [35]:
## simple promotion flag creation 

In [36]:
model_df["has_promotion"] = (model_df["onpromotion"] > 0).astype(int)

In [37]:
model_df[["onpromotion", "has_promotion"]].head(10)

,onpromotion,has_promotion
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0
5,0,0
6,0,0
7,0,0
8,0,0
9,0,0


## lag features Creation 

In [38]:
model_df["lag_1"] = model_df.groupby(["store_nbr", "family"])["sales"].shift(1)
model_df["lag_7"] = model_df.groupby(["store_nbr", "family"])["sales"].shift(7)
model_df["lag_14"] = model_df.groupby(["store_nbr", "family"])["sales"].shift(14)

C:\Users\hp\AppData\Local\Temp\ipykernel_8144\4014734819.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  model_df["lag_1"] = model_df.groupby(["store_nbr", "family"])["sales"].shift(1)
C:\Users\hp\AppData\Local\Temp\ipykernel_8144\4014734819.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  model_df["lag_7"] = model_df.groupby(["store_nbr", "family"])["sales"].shift(7)
C:\Users\hp\AppData\Local\Temp\ipykernel_8144\4014734819.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observe

In [39]:
model_df[["store_nbr", "family", "date", "sales", "lag_1", "lag_7", "lag_14"]].head(15)

,store_nbr,family,date,sales,lag_1,lag_7,lag_14
0,1,AUTOMOTIVE,2013-01-01,0.0,NaN,NaN,NaN
1,1,AUTOMOTIVE,2013-01-02,2.0,0.0,NaN,NaN
2,1,AUTOMOTIVE,2013-01-03,3.0,2.0,NaN,NaN
3,1,AUTOMOTIVE,2013-01-04,3.0,3.0,NaN,NaN
4,1,AUTOMOTIVE,2013-01-05,5.0,3.0,NaN,NaN
5,1,AUTOMOTIVE,2013-01-06,2.0,5.0,NaN,NaN
6,1,AUTOMOTIVE,2013-01-07,0.0,2.0,NaN,NaN
7,1,AUTOMOTIVE,2013-01-08,2.0,0.0,0.0,NaN
8,1,AUTOMOTIVE,2013-01-09,2.0,2.0,2.0,NaN
9,1,AUTOMOTIVE,2013-01-10,2.0,2.0,3.0,NaN


## Rolling Averages

In [40]:
model_df["rolling_mean_7"] = (
    model_df.groupby(["store_nbr", "family"])["sales"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

model_df["rolling_mean_14"] = (
    model_df.groupby(["store_nbr", "family"])["sales"]
    .transform(lambda x: x.shift(1).rolling(14).mean())
)

model_df["rolling_mean_28"] = (
    model_df.groupby(["store_nbr", "family"])["sales"]
    .transform(lambda x: x.shift(1).rolling(28).mean())
)

C:\Users\hp\AppData\Local\Temp\ipykernel_8144\605239310.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  model_df.groupby(["store_nbr", "family"])["sales"]
C:\Users\hp\AppData\Local\Temp\ipykernel_8144\605239310.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  model_df.groupby(["store_nbr", "family"])["sales"]
C:\Users\hp\AppData\Local\Temp\ipykernel_8144\605239310.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

In [41]:
model_df[[
    "store_nbr", "family", "date", "sales",
    "rolling_mean_7", "rolling_mean_14", "rolling_mean_28"
]].head(20)

,store_nbr,family,date,sales,rolling_mean_7,rolling_mean_14,rolling_mean_28
0,1,AUTOMOTIVE,2013-01-01,0.0,NaN,NaN,NaN
1,1,AUTOMOTIVE,2013-01-02,2.0,NaN,NaN,NaN
2,1,AUTOMOTIVE,2013-01-03,3.0,NaN,NaN,NaN
3,1,AUTOMOTIVE,2013-01-04,3.0,NaN,NaN,NaN
4,1,AUTOMOTIVE,2013-01-05,5.0,NaN,NaN,NaN
5,1,AUTOMOTIVE,2013-01-06,2.0,NaN,NaN,NaN
6,1,AUTOMOTIVE,2013-01-07,0.0,NaN,NaN,NaN
7,1,AUTOMOTIVE,2013-01-08,2.0,2.142857,NaN,NaN
8,1,AUTOMOTIVE,2013-01-09,2.0,2.428571,NaN,NaN
9,1,AUTOMOTIVE,2013-01-10,2.0,2.428571,NaN,NaN


# rolling volatility

In [42]:
model_df["rolling_std_7"] = (
    model_df.groupby(["store_nbr", "family"])["sales"]
    .transform(lambda x: x.shift(1).rolling(7).std())
)

model_df["rolling_std_28"] = (
    model_df.groupby(["store_nbr", "family"])["sales"]
    .transform(lambda x: x.shift(1).rolling(28).std())
)

C:\Users\hp\AppData\Local\Temp\ipykernel_8144\3238018095.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  model_df.groupby(["store_nbr", "family"])["sales"]
C:\Users\hp\AppData\Local\Temp\ipykernel_8144\3238018095.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  model_df.groupby(["store_nbr", "family"])["sales"]


In [43]:
model_df["avg_daily_sales_7"] = model_df["rolling_mean_7"]
model_df["avg_daily_sales_14"] = model_df["rolling_mean_14"]

In [44]:
# lack of data so results are normal 
model_df[[
    "lag_1", "lag_7", "lag_14",
    "rolling_mean_7", "rolling_mean_14", "rolling_mean_28",
    "rolling_std_7", "rolling_std_28"
]].isnull().sum()

lag_1               1782
lag_7              12474
lag_14             24948
rolling_mean_7     12474
rolling_mean_14    24948
rolling_mean_28    49896
rolling_std_7      12474
rolling_std_28     49896
dtype: int64

## supplier lead time 

In [45]:
family_lead_time_map = {
    "GROCERY I": 3,
    "BEVERAGES": 4,
    "PRODUCE": 2,
    "DAIRY": 3,
    "CLEANING": 5
}

model_df["supplier_lead_time"] = model_df["family"].map(family_lead_time_map)
model_df["supplier_lead_time"] = model_df["supplier_lead_time"].fillna(4)

## Stock on hand

In [46]:
model_df["simulated_stock_on_hand"] = (
    model_df["avg_daily_sales_14"].fillna(0) * 10
).round(0)

## reorder point creation

In [47]:
model_df["reorder_point"] = (
    model_df["avg_daily_sales_14"].fillna(0) * model_df["supplier_lead_time"] * 1.2
).round(0)

## days until stockout

In [48]:
import numpy as np

model_df["days_until_stockout"] = np.where(
    model_df["avg_daily_sales_7"] > 0,
    model_df["simulated_stock_on_hand"] / model_df["avg_daily_sales_7"],
    np.nan
)

model_df["days_until_stockout"] = model_df["days_until_stockout"].replace([np.inf, -np.inf], np.nan)

## Stockout risk flag 

In [49]:
model_df["stockout_risk_flag"] = (
    model_df["days_until_stockout"] < 7
).astype(int)

## reorder urgency score

In [50]:
model_df["reorder_urgency_score"] = (
    model_df["reorder_point"] - model_df["simulated_stock_on_hand"]
)

## new columns inspection

In [51]:
model_df[[
    "date", "store_nbr", "family", "sales",
    "avg_daily_sales_7", "avg_daily_sales_14",
    "supplier_lead_time", "simulated_stock_on_hand",
    "reorder_point", "days_until_stockout",
    "stockout_risk_flag", "reorder_urgency_score"
]].head(20)

,date,store_nbr,family,sales,avg_daily_sales_7,avg_daily_sales_14,supplier_lead_time,simulated_stock_on_hand,reorder_point,days_until_stockout,stockout_risk_flag,reorder_urgency_score
0,2013-01-01,1,AUTOMOTIVE,0.0,NaN,NaN,4.0,0.0,0.0,NaN,0,0.0
1,2013-01-02,1,AUTOMOTIVE,2.0,NaN,NaN,4.0,0.0,0.0,NaN,0,0.0
2,2013-01-03,1,AUTOMOTIVE,3.0,NaN,NaN,4.0,0.0,0.0,NaN,0,0.0
3,2013-01-04,1,AUTOMOTIVE,3.0,NaN,NaN,4.0,0.0,0.0,NaN,0,0.0
4,2013-01-05,1,AUTOMOTIVE,5.0,NaN,NaN,4.0,0.0,0.0,NaN,0,0.0
5,2013-01-06,1,AUTOMOTIVE,2.0,NaN,NaN,4.0,0.0,0.0,NaN,0,0.0
6,2013-01-07,1,AUTOMOTIVE,0.0,NaN,NaN,4.0,0.0,0.0,NaN,0,0.0
7,2013-01-08,1,AUTOMOTIVE,2.0,2.142857,NaN,4.0,0.0,0.0,0.000000,1,0.0
8,2013-01-09,1,AUTOMOTIVE,2.0,2.428571,NaN,4.0,0.0,0.0,0.000000,1,0.0
9,2013-01-10,1,AUTOMOTIVE,2.0,2.428571,NaN,4.0,0.0,0.0,0.000000,1,0.0


## saving feature dataset

In [52]:
import os
os.makedirs("data/processed", exist_ok=True)

model_df.to_csv("data/processed/modeling_data_features.csv", index=False)